# 🔮 UIDAI Aadhaar Biometric Analysis, Model Comparison & Planning Pipeline

This notebook outlines the end-to-end data analytics and machine learning pipeline developed for the UIDAI Hackathon. It contains:
1. **Data Cleaning & Standardization** (resolving spelling and whitespace issues across 36 states)
2. **Feature Engineering** (building lags, rolling metrics, capacity load ratios)
3. **Exploratory Data Analysis (EDA)** (visualizing state-wise distributions and seasonality)
4. **Model Training & Comparison** (comparing Baseline, Random Forest, and LightGBM models on future test sets)
5. **Operational Analytics** (defining the logic for the 'What-If' Resource Simulator)
6. **Model Serialization** (saving binaries for the Streamlit dashboard)

In [ ]:
import pandas as pd
import numpy as np
import glob
import joblib
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import lightgbm as lgb
    USE_LGB = True
except ImportError:
    print("LightGBM not installed. We will train RandomForest only.")
    USE_LGB = False

## 1. Data Cleaning & Standardization

We merge all CSV parts and clean state and district names. The raw dataset contains spelling differences and extra spaces (generating 57 states). We normalize them to the standard 36 states/Union Territories of India.

In [ ]:
# Find all source CSV files
csv_files = glob.glob("data/api_data_aadhar_biometric_*.csv")
print("Found source CSV files:", csv_files)

dfs = []
for f in csv_files:
    print(f"Reading {f}...")
    dfs.append(pd.read_csv(f))
    
df = pd.concat(dfs, ignore_index=True)
print(f"Merged total rows: {len(df):,}")

# Clean strings and correct formatting
df['state'] = df['state'].astype(str).str.strip().str.title()
df['district'] = df['district'].astype(str).str.strip().str.title()

# Mapping dictionary for states/UTs
state_mapping = {
    'Andaman And Nicobar Islands': 'Andaman & Nicobar Islands',
    'Chhatisgarh': 'Chhattisgarh',
    'Dadra & Nagar Haveli': 'Dadra & Nagar Haveli and Daman & Diu',
    'Dadra And Nagar Haveli': 'Dadra & Nagar Haveli and Daman & Diu',
    'Dadra And Nagar Haveli And Daman And Diu': 'Dadra & Nagar Haveli and Daman & Diu',
    'Daman & Diu': 'Dadra & Nagar Haveli and Daman & Diu',
    'Daman And Diu': 'Dadra & Nagar Haveli and Daman & Diu',
    'Jammu And Kashmir': 'Jammu & Kashmir',
    'Orissa': 'Odisha',
    'Pondicherry': 'Puducherry',
    'Tamilnadu': 'Tamil Nadu',
    'Uttaranchal': 'Uttarakhand',
    'West  Bengal': 'West Bengal',
    'West Bangal': 'West Bengal',
    'Westbengal': 'West Bengal'
}
df['state'] = df['state'].replace(state_mapping)
df['date'] = pd.to_datetime(df['date'].str.strip(), format='%d-%m-%Y')

# Sort chronologically and spatially
df = df.sort_values(by=['date', 'state', 'district', 'pincode']).reset_index(drop=True)

# Save cleaned data directly to root folder
df.to_csv("data/cleaned_biometric_data.csv", index=False)
print(f"After standardization: {df['state'].nunique()} unique states, {df['district'].nunique()} unique districts.")

## 2. Feature Engineering

We roll up the pincode-level daily counts to the district level. Then we compute lag and rolling values to capture temporal trends, capacity load risk thresholds, and apply anomaly models.

In [ ]:
print("Aggregating pincode hotspots summary...")
pincode_summary = df.groupby(['state', 'district', 'pincode']).agg(
    total_bio_5_17=('bio_age_5_17', 'sum'),
    total_bio_17=('bio_age_17_', 'sum')
).reset_index()
pincode_summary['total_updates'] = pincode_summary['total_bio_5_17'] + pincode_summary['total_bio_17']
pincode_summary.to_csv("data/pincode_hotspots.csv", index=False)

# Aggregate at District level
df['total_bio'] = df['bio_age_5_17'] + df['bio_age_17_']
district_df = df.groupby(['date', 'state', 'district']).agg({
    'bio_age_5_17': 'sum',
    'bio_age_17_': 'sum',
    'total_bio': 'sum'
}).reset_index()

district_df = district_df.sort_values(by=['state', 'district', 'date']).reset_index(drop=True)

# Time-series Lags
district_df['total_bio_lag_7'] = district_df.groupby(['state', 'district'])['total_bio'].shift(7)
district_df['total_bio_lag_14'] = district_df.groupby(['state', 'district'])['total_bio'].shift(14)

# Rolling Statistics
district_df['total_bio_roll_mean_7'] = district_df.groupby(['state', 'district'])['total_bio'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)
district_df['total_bio_roll_mean_30'] = district_df.groupby(['state', 'district'])['total_bio'].transform(
    lambda x: x.rolling(window=30, min_periods=1).mean()
)
district_df['total_bio_roll_std_30'] = district_df.groupby(['state', 'district'])['total_bio'].transform(
    lambda x: x.rolling(window=30, min_periods=1).std()
).fillna(0)

# Relative Capacity & Risk labels
district_df['capacity_ratio'] = district_df['total_bio'] / (district_df['total_bio_roll_mean_30'] + 1)
district_df['risk_level'] = 'Low'
district_df.loc[district_df['capacity_ratio'] > 1.0, 'risk_level'] = 'Medium'
district_df.loc[district_df['capacity_ratio'] > 1.3, 'risk_level'] = 'High'

# Anomaly detection (Z-score)
district_df['z_score'] = (district_df['total_bio'] - district_df['total_bio_roll_mean_30']) / (district_df['total_bio_roll_std_30'] + 1)
district_df['is_anomaly_z'] = ((district_df['z_score'] > 3.0) & (district_df['total_bio'] > 100)).astype(int)

# Anomaly detection (Isolation Forest)
anomaly_features = ['total_bio', 'bio_age_5_17', 'bio_age_17_', 'capacity_ratio', 'z_score']
X_anomaly = district_df[anomaly_features].fillna(0)
iso_forest = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
district_df['is_anomaly_if'] = (iso_forest.fit_predict(X_anomaly) == -1).astype(int)

# Date segments
district_df['day'] = district_df['date'].dt.day
district_df['month'] = district_df['date'].dt.month
district_df['weekday'] = district_df['date'].dt.weekday
district_df['year'] = district_df['date'].dt.year

district_df.to_csv("data/district_features.csv", index=False)
print(f"Created district features dataframe with shape {district_df.shape}")

## 3. Exploratory Data Analysis & Visualizations

We analyze historical patterns, total state transaction counts, and seasonal changes.

In [ ]:
# 1. Plot top 10 states by total updates
state_totals = district_df.groupby('state')['total_bio'].sum().reset_index()
state_totals = state_totals.sort_values(by='total_bio', ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(data=state_totals, x='total_bio', y='state', palette='Blues_r')
plt.title('Top 10 States by Total Biometric Updates')
plt.xlabel('Total Volume of Updates')
plt.ylabel('State')
plt.tight_layout()
plt.savefig('plots/top_10_states_updates.png', dpi=150)
plt.show()

# 2. Plot monthly seasonality
monthly_trend = district_df.groupby('month')['total_bio'].sum().reset_index()
plt.figure(figsize=(8, 4))
sns.lineplot(data=monthly_trend, x='month', y='total_bio', marker='o', color='teal', linewidth=2.5)
plt.title('Monthly Distribution of Biometric Updates')
plt.xlabel('Month')
plt.ylabel('Updates')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('plots/monthly_seasonality.png', dpi=150)
plt.show()

## 4. Model Training & Comparison

We train our LightGBM forecasting model. To avoid temporal data leakage, we train on data before November 15, 2025, and test on the subsequent data.

In [ ]:
# Clean null lag rows
df_clean = district_df.dropna(subset=['total_bio_lag_7', 'total_bio_lag_14', 'total_bio_roll_mean_30']).copy()

cat_features = ['state', 'district']
num_features = [
    'total_bio_lag_7', 'total_bio_lag_14', 
    'total_bio_roll_mean_7', 'total_bio_roll_mean_30', 'total_bio_roll_std_30',
    'day', 'month', 'weekday'
]
features = cat_features + num_features
target = 'total_bio'

# Set category data type
for col in cat_features:
    df_clean[col] = df_clean[col].astype('category')

split_date = pd.to_datetime("2025-11-15")
train_mask = df_clean['date'] <= split_date
test_mask = df_clean['date'] > split_date

X_train = df_clean.loc[train_mask, features]
y_train = df_clean.loc[train_mask, target]
X_test = df_clean.loc[test_mask, features]
y_test = df_clean.loc[test_mask, target]

print(f"Training set: {X_train.shape[0]:,} rows | Test set: {X_test.shape[0]:,} rows")

# --- Model 1: Naive Rolling 7-day Baseline ---
y_pred_baseline = df_clean.loc[test_mask, 'total_bio_roll_mean_7']
mae_baseline = mean_absolute_error(y_test, y_pred_baseline)
r2_baseline = r2_score(y_test, y_pred_baseline)

# --- Model 2: Random Forest ---
print("Training Random Forest Regressor...")
X_train_rf = X_train.copy()
X_test_rf = X_test.copy()
for col in cat_features:
    X_train_rf[col] = X_train_rf[col].cat.codes
    X_test_rf[col] = X_test_rf[col].cat.codes

rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train_rf, y_train)
y_pred_rf = rf_model.predict(X_test_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

# --- Model 3: LightGBM Regressor ---
if USE_LGB:
    print("Training LightGBM Regressor...")
    lgb_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=42, n_jobs=-1)
    lgb_model.fit(X_train, y_train, categorical_feature=cat_features)
    y_pred_lgb = lgb_model.predict(X_test)
    mae_lgb = mean_absolute_error(y_test, y_pred_lgb)
    r2_lgb = r2_score(y_test, y_pred_lgb)
    best_model = lgb_model
else:
    mae_lgb = np.nan
    r2_lgb = np.nan
    best_model = rf_model

# Compile Performance Table
comparison_data = {
    'Model': ['7-Day Rolling Baseline', 'Random Forest Regressor', 'LightGBM Regressor'],
    'MAE (Lower is Better)': [mae_baseline, mae_rf, mae_lgb],
    'R-squared (R2) Score': [r2_baseline, r2_rf, r2_lgb]
}
results_df = pd.DataFrame(comparison_data)
print("\n--- Model Performance Comparison ---")
print(results_df.to_string(index=False))

### Performance Visualization

In [ ]:
# Plot comparison
plt.figure(figsize=(10, 4.5))
plt.subplot(1, 2, 1)
sns.barplot(data=results_df, x='Model', y='MAE (Lower is Better)', palette='muted')
plt.title('MAE Comparison (Lower is Better)')
plt.xticks(rotation=15)

plt.subplot(1, 2, 2)
sns.barplot(data=results_df, x='Model', y='R-squared (R2) Score', palette='muted')
plt.title('R-squared Comparison (Higher is Better)')
plt.xticks(rotation=15)

plt.tight_layout()
plt.savefig('plots/model_performance_comparison.png', dpi=150)
plt.show()

## 5. Prescriptive Operational Simulator Logic

To make predictions actionable, we define a **What-If Planner** equation:

$$\text{Capacity} = (\text{Centers} \times \text{Kits} \times \text{Daily Kit Rate}) + (\text{Mobile Vans} \times \text{Van Rate})$$
$$\text{Congestion Index} = \frac{\text{Forecasted Demand}}{\text{Capacity}}$$

If $\text{Congestion Index} > 1.0$, a capacity deficit exists and the dashboard dynamically recommends van deployment to prevent delays.

## 6. Serializing Models for Dashboard Deployment

We save the models and encoders directly in the root workspace folder to enable Streamlit dashboard loading.

In [ ]:
# Save category mappings directly to root
category_mappings = {col: list(df_clean[col].cat.categories) for col in cat_features}
joblib.dump(category_mappings, "models/category_mappings.pkl")

# Save trained forecaster directly to root
joblib.dump(best_model, "models/demand_forecaster.pkl")

# Save trained Isolation Forest anomaly detector directly to root
joblib.dump(iso_forest, "models/anomaly_detector.pkl")

print("All model files and category structures exported successfully to current directory for app deployment!")